<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 4 · Unsupervised Learning and Real Projects · Lesson 20 of 24

# PCA: Seeing Many Features in Two

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Explain what principal component analysis does, read explained variance, plot high-dimensional data in 2-D, and use PCA inside a model pipeline.

## Too many features to look at

The wine dataset in scikit-learn has **13** chemical measurements for 178 wines from three grape varieties (cultivars). You cannot draw a 13-dimensional chart. **Principal component analysis (PCA)** finds new axes — **principal components** — that keep as much of the spread (variance) as possible in as few dimensions as possible.

- **PC1** is the direction in which the data varies most.
- **PC2** is the next most varied direction, at a right angle to PC1.
- And so on. Each component is a **weighted mix** of the original features.

It is like taking the best photo of a 3-D object: you choose the angle that shows the most detail.

## PCA on the wine data

PCA is driven by variance, so features must be **standardised** first — otherwise the feature with the biggest units (proline, in the hundreds) would dominate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine = load_wine(as_frame=True)
X_wine = StandardScaler().fit_transform(wine.data)

pca = PCA().fit(X_wine)
print(pca.explained_variance_ratio_.round(3))
print(pca.explained_variance_ratio_.cumsum().round(3))

- PC1 explains about **36%** of the variance and PC2 about **19%**: together about **55%** with only two numbers per wine instead of 13.
- About 10 components are needed to keep 95% of the variance.

## Plot 13 features in 2-D

In [ ]:
pca2 = PCA(n_components=2)
coords = pca2.fit_transform(X_wine)
plt.scatter(coords[:, 0], coords[:, 1], c=wine.target, s=15)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("178 wines, 13 measurements, shown in 2-D")
plt.show()

The three varieties form three clearly separated clouds — even though PCA **never saw the labels**. That shows the 13 measurements really contain the information needed to tell them apart.

## What does a component mean?

In [ ]:
loadings = pd.Series(pca2.components_[0], index=wine.data.columns).sort_values()
print(loadings.round(2))

PC1 has large positive weights for flavanoids and total phenols and negative weights for non-flavanoid phenols and malic acid — so PC1 is roughly a "phenol richness" axis. Components are mixtures, so their meaning is often fuzzy. That is the price of PCA.

## PCA before a model

PCA can make data smaller and remove noise before a model. Let's try it on the breast-cancer data (30 features):

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

cancer = load_breast_cancer()
for n in [2, 5, 10]:
    pipe = make_pipeline(StandardScaler(), PCA(n_components=n), LogisticRegression(max_iter=1000))
    print(n, "components:", cross_val_score(pipe, cancer.data, cancer.target, cv=5).mean().round(3))
full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print("all 30 features:", cross_val_score(full, cancer.data, cancer.target, cv=5).mean().round(3))

With just **2** components the model already scores about **0.95**; with 10 it matches all 30 features (about **0.98**). You can also ask for a share of variance: `PCA(n_components=0.95)` keeps as many components as needed for 95%.

## When to use PCA

| Good for | Be careful |
|---|---|
| Visualising many features | Components are hard to explain |
| Compressing data, speeding up models | Throws away some information |
| Removing correlated, noisy features | Needs scaling first |
| Image and gene data with thousands of features | Tree models rarely need it |

For pictures of complex data, **t-SNE** and **UMAP** often show clusters more clearly, but they are only for visualisation.

## Key terms

- **Dimensionality reduction** — representing data with fewer features.
- **Principal component** — a new axis that captures as much variance as possible.
- **Explained variance ratio** — the share of the total variance kept by each component.
- **Loadings** — the weights of the original features in a component.

**Takeaway:** PCA rotates the data so the first few components keep most of the information; it is excellent for 2-D pictures and compression, at the cost of easy interpretation.

## Try it yourself

1. Plot the cumulative explained variance of the wine data (`plt.plot(...cumsum())`) and mark the 90% line.
2. Run PCA on the wine data **without** scaling. How much does PC1 explain now, and which feature dominates it? Why?
3. Use `PCA(n_components=0.95)` in the cancer pipeline. How many components does it keep (`pipe[1].n_components_`)?
4. Run PCA with 2 components on the customer data from Lesson 19 and colour the plot by k-means segment.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*